# ViT PPO Fine-tuning — Kaggle

Дообучает трансформер (SmallViT) через PPO начиная с BC-весов.

**Перед запуском:**
1. Загрузи `vit_bc_pretrained.zip` на Kaggle Datasets
2. Добавь датасет через Add Input
3. Accelerator: T4 x2
4. Internet: On

In [ ]:
# Клонируем репо и устанавливаем зависимости
# torch не указываем — на Kaggle уже установлен
!git clone https://github.com/Andrew82mm/RL_practice.git /kaggle/working/RL_practice
%cd /kaggle/working/RL_practice
!pip install -q sb3-contrib gymnasium scipy tqdm

In [ ]:
import torch, os
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не найден!'}")
%cd /kaggle/working/RL_practice

In [ ]:
import os, shutil, zipfile

BC_PATH = "/kaggle/working/vit_bc_pretrained.zip"
found_zip = None
found_dir = None

def _is_vit_pretrained(name):
    # Kaggle заменяет '_' на '-' в именах папок
    n = name.lower().replace("-", "_")
    return "vit_bc_pretrained" in n

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if _is_vit_pretrained(f) and f.endswith(".zip"):
            found_zip = os.path.join(root, f)
    for d in dirs:
        if _is_vit_pretrained(d):
            found_dir = os.path.join(root, d)

if found_zip:
    print(f"Найден zip: {found_zip}")
    shutil.copy2(found_zip, BC_PATH)
elif found_dir:
    print(f"Kaggle распаковал архив, перепаковываем из: {found_dir}")
    with zipfile.ZipFile(BC_PATH, "w") as zf:
        for f in os.listdir(found_dir):
            zf.write(os.path.join(found_dir, f), f)
else:
    print("Содержимое /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        print(f"  {root}: {files}")
    raise FileNotFoundError("vit_bc_pretrained не найден! Проверь что датасет добавлен через Add Input")

print(f"Готово: {BC_PATH} ({os.path.getsize(BC_PATH)/1024**2:.1f} MB)")

In [ ]:
# Запускаем PPO с ViT + BC-инициализацией
# --timesteps 15000000  — 15M шагов (~8-10 часов на T4)
# --n-envs 8           — не перегружаем 2 CPU ядра Kaggle
!python training/run_training.py \
    --arch vit \
    --bc-pretrained /kaggle/working/vit_bc_pretrained.zip \
    --timesteps 15000000 \
    --n-envs 8 \
    --run-name vit_bc_finetune

In [ ]:
import os

MODEL_DIR = "/kaggle/working/RL_practice/models"
print(f"Модели в {MODEL_DIR}:")
for root, dirs, files in os.walk(MODEL_DIR):
    for f in files:
        path = os.path.join(root, f)
        print(f"  {path}  ({os.path.getsize(path)/1024**2:.1f} MB)")